# Notebook 13 – Missing Values
## Learning objectives
Detect, remove, fill, and interpolate missing values, then choose an appropriate handling strategy.

In [3]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "customer": ["Asha", "Ben", "Chen", "Divya", "Eli"],
    "age": [24, np.nan, 31, np.nan, 28],
    "city": ["Delhi", "Mumbai", None, "Delhi", "Chennai"],
    "daily_sales": [100.0, np.nan, 130.0, np.nan, 150.0]
})
print(df)

  customer   age     city  daily_sales
0     Asha  24.0    Delhi        100.0
1      Ben   NaN   Mumbai          NaN
2     Chen  31.0      NaN        130.0
3    Divya   NaN    Delhi          NaN
4      Eli  28.0  Chennai        150.0


## 1. `isnull()`
**Definition:** Returns `True` for missing values and `False` for present values.
**Real-world:** Find incomplete survey records.
**Business:** Check which customer records lack a city.
**AI/ML:** Measure missingness before preprocessing.

In [4]:
print(df.isnull())
print("\\nMissing values per column:")
print(df.isnull().sum())

   customer    age   city  daily_sales
0     False  False  False        False
1     False   True  False         True
2     False  False   True        False
3     False   True  False         True
4     False  False  False        False
\nMissing values per column:
customer       0
age            2
city           1
daily_sales    2
dtype: int64


## 2. `notnull()`
**Definition:** Returns `True` for values that are not missing.
**Real-world:** Find completed form fields.
**Business:** Select rows with a known city.
**AI/ML:** Filter records where a required feature is available.

In [5]:
print(df["city"].notnull())
print("\\nRows with a known city:")
print(df[df["city"].notnull()])

0     True
1     True
2    False
3     True
4     True
Name: city, dtype: bool
\nRows with a known city:
  customer   age     city  daily_sales
0     Asha  24.0    Delhi        100.0
1      Ben   NaN   Mumbai          NaN
3    Divya   NaN    Delhi          NaN
4      Eli  28.0  Chennai        150.0


## 3. `dropna()`
**Definition:** Removes rows or columns containing missing values, depending on the arguments.
**Real-world:** Remove unusable records from a small report.
**Business:** Create a report from rows with all required fields.
**AI/ML:** Drop records only when the missingness makes them unsuitable.

Warning: dropping data can create bias or remove valuable information. Choose the subset and axis deliberately.

In [6]:
print("Rows with no missing values:")
print(df.dropna())
print("\\nRows where age is present:")
print(df.dropna(subset=["age"]))

Rows with no missing values:
  customer   age     city  daily_sales
0     Asha  24.0    Delhi        100.0
4      Eli  28.0  Chennai        150.0
\nRows where age is present:
  customer   age     city  daily_sales
0     Asha  24.0    Delhi        100.0
2     Chen  31.0      NaN        130.0
4      Eli  28.0  Chennai        150.0


## 4. `fillna()`
**Definition:** Replaces missing values with a chosen value or a value derived from the data.
**Real-world:** Fill a missing optional label with `Unknown`.
**Business:** Fill missing sales with zero only when missing truly means no sales.
**AI/ML:** Impute numeric features with a median or categorical features with a mode.

Do not automatically replace every missing number with zero; zero may have a different meaning from “unknown.”

In [ ]:
filled = df.copy()
filled["age"] = filled["age"].fillna(filled["age"].median())
filled["city"] = filled["city"].fillna("Unknown")
print(filled)

## 5. `interpolate()`
**Definition:** Estimates missing values from nearby values, often useful for ordered numeric time series.
**Real-world:** Estimate a missing temperature reading between two measurements.
**Business:** Fill a short gap in hourly energy usage.
**AI/ML:** Prepare continuous sensor signals, when interpolation is appropriate.

In [ ]:
readings = pd.DataFrame({
    "hour": [1, 2, 3, 4, 5],
    "temperature": [20.0, np.nan, 24.0, np.nan, 28.0]
})
readings["interpolated"] = readings["temperature"].interpolate()
print(readings)

## 6. Missing Value Handling Strategies
**Definition:** A strategy is a reasoned choice about what to do with missing data. Common options include:
- **Drop rows/columns:** when the data is unusable and removal is justified.
- **Mean/median imputation:** numeric features; median is less affected by extreme values.
- **Mode or `Unknown`:** categorical features.
- **Forward/backward fill:** ordered data when carrying the last/next observation is meaningful.
- **Interpolation:** smooth, ordered numeric series.
- **Missingness indicator:** add a column showing whether a value was missing.
- **Model-based imputation:** use other variables to estimate missing values, with care.

**Business example:** A missing customer age might be imputed with the median, while a missing city may be labeled `Unknown`.
**AI/ML example:** Fit imputers on training data only, then apply the same fitted transformation to validation/test data to avoid data leakage.

In [ ]:
strategy_df = df.copy()
strategy_df["age_was_missing"] = strategy_df["age"].isnull().astype(int)
strategy_df["age"] = strategy_df["age"].fillna(strategy_df["age"].median())
strategy_df["city"] = strategy_df["city"].fillna("Unknown")
print(strategy_df)

# Example of forward fill for ordered readings:
print(readings["temperature"].ffill())